# Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

In [3]:
file_path = "/content/drive/MyDrive/cleaned_file.xlsx"

In [4]:
df = pd.read_excel(file_path)

In [5]:
df.columns

Index(['ID', 'Date', 'Content', 'Type', 'Org. type', 'Org. name', 'Country',
       'Company name confidential', 'Attachment', 'Privacy statement',
       'General Comments', 'Answer to specific info request 1',
       'Answer to specific info request 2',
       'Answer to specific info request 3',
       'Answer to specific info request 4',
       'Answer to specific info request 5',
       'Answer to specific info request 6',
       'Answer to specific info request 7',
       'Answer to specific info request 8',
       'Answer to specific info request 9',
       'Answer to specific info request 10', 'Source File', 'language',
       'attachment_author', 'attachments', 'attachment_link_dummy',
       'confidential_info_dummy', 'missing_key_info', 'combined_text',
       'cleaned_text'],
      dtype='object')

# Stance detection

In [7]:
API_ENDPOINT = "..."
API_KEY = "..."

In [8]:
import openai

In [9]:
client = openai.OpenAI(
    api_key= API_KEY,
    base_url= API_ENDPOINT)

In [11]:
input_path = "/content/drive/MyDrive/cleaned_file.xlsx"
output_path = "/content/drive/MyDrive/pfas_stances.xlsx"
checkpoint_interval = 5

In [5]:
import pandas as pd
import openai
import time
import os
from tqdm import tqdm

## Few-shot stance detection: grouped stances

In [14]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "Stance"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Starting from input


In [15]:
comment_1 = """I support the limitation proposal as it is and hope that it will be adopted as it looks today. PFAS have proven to be toxic and persistent chemicals that negatively affect both human health and the environment, and I therefore only see it as a positive thing that their use is being stopped, especially in products where they are not essential. For example, in makeup, electronics, or in clothes! I believe that it is more beneficial for humanity in the long run if the quality of products potentially decreases slightly (e.g. a raincoat not repelling water as well) than if we become more exposed to higher concentrations of persistent chemicals like PFAS."""
comment_2 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_3 = """Over the last years, perfluorocarbons and perfluoropolyethers have been frequently used as dielectric fluids for immersion cooling applications. The use of such liquids entails a high environmental impact as well as severe health risks during operation, maintenance and handling of the immersed equipment. Recently, new and more sustainable dielectric liquids – able to efficiently operate these systems - have entered the market, resulting in no need for per- and polyfluoroalkyl substances anymore. 9165 Cleaning and heat transfer: engineered fluids. Electronics: heat transfer fluid for (single-phase) immersion cooling (in EV, IT equipment). 9165       PFAS alternatives for (single-phase) immersion cooling applications. Recently, sustainable hydrocarbon (e.g. DC COOLING BioLife 4 from TotalEnergies) and ester (e.g. NatureCool2000 from Cargill, Mivolt DF7 from M&I Materials) based dielectric liquids have entered the immersion cooling market."""

comment_4 = """We acknowledge  the  necessity  of  regulating  PFAS,  but  the  current  restriction proposal is too broad and too impacting to be relevant.  Fluoropolymers (including fluoroelastomers)  should  be  out  of  scope  of  this  restriction,  and  the  “repair  as produced” principle should be respected for all existing vehicles. Fluoropolymers (including fluoroelastomers) were quest their removal from the scope of the restriction. Concerning the manufacturing phase, the risks of PFAS emissions to the environment can be controlled with alternative Risk Management Options. Concerning the use phase, they are considered non- toxic, non-bioaccumulative, non-mobile and as such, are classed as polymers of low concern. Concerning the end-of-life phase, incineration of fluoropolymers does not contribute to environmental PFAS emissions and is a safe method of disposal. 7296       The PFAS REACH restriction is expected to have a major impact on the automotive industry. The automotive industry is a major downstream user of many PFAS, including fluoropolymers, fluorinated gases, and short-chain PFAS. Fluoropolymers are used for several key technical components, such as gaskets, hoses, joints, O-rings, seals, cords, cables, or sleeves. The current proposal does not acknowledge any derogations for such uses, whereas alternatives are not readily available and do not share sufficient properties to be qualified. In the context of this derogation, the automotive industry wishes to express its great concern if the implementation of the restriction were to continue as is and proposes an alternative implementation approach that integrates the technical and economic constraints on the one hand and preserves the objectives of electromobility on the other."""
comment_5 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_6 = """As engineering service provider, planner and constructor for large-scale plant construction, we specify and handle various materials and material groups. Among others, the following semi-finished products and assemblies made of fluoropolymers are used in our range of activities: - Fluoropolymers as sealing materials (flat, O-ring, mechanical seal, etc.) in various equipment/pipelines - Fluoropolymers as lining materials in pumps and pipeline materials - Fluoropolymers as membrane materials in pumps, electrolyzers Additionally, fluoropolymers are indirectly used as coolants or additives in associated processes, which however fall within the scope of the end user/customer. As the broad field of mechanical and plant engineering is not listed in the sectors mentioned in Annex XV (Table 9) of the restriction report, the following areas were selected as the most relevant - Energy sector (Annex E.2.12.) - Sector as a whole - Petroleum and mining (Annex E.2.15.) - Fluoropolymer applications Furthermore, see SECTION III. Non-confidential comments. 9567" "We, as large-scale plant builders, can divide the percentage emission shares for our scope as follows: 0% emissions in the production phase (as we appear as end users) 0% emissions in the usage phase 100% emissions in the end-of-life phase (this is outside our scope and not considered) 9567"    "See SECTION III. Non-confidential comments"""

comment_7 = """The European Society for Medical Oncology (ESMO) notes the draft proposal’s aims to prevent PFAS accumulation in the environment and food chain and welcomes its efforts to improve human health. However, it is pivotal that such action is based on a comprehensive and consistent review of the available evidence as there are growing concerns about the possible impact of the draft proposal on the production, availability and manufacturing of cancer medicines. 9551 Pharmaceuticals, medical devices, excipients, tarting materials and chemical intermediates, reagents, solvents, catalysts, auxiliaries. """
comment_8 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""
comment_9 = """Integrated DNA Technologies, a Biotechnology company, develops, manufactures, and markets nucleic acid (DNA and RNA) products that support the life sciences industry. IDT supplies synthetic biology products used in genomics research, diagnostics, specialized reagents, consumables, and medical device products. Our customers use our products in applications ranging from fundamental biological and genomics research to making lifesaving vaccines, biologic drugs, and novel cell and gene therapies. We supply the tools and services that help our customers do their work better, faster, and safer. IDT designs, builds, and then deploys proprietary manufacturing equipment into the European Union to manufacture nucleic acid products. We have over 1400 associates across nine countries worldwide including manufacturing sites in Belgium, Singapore, and the USA.  General Comments According to ECHA’s Press Release, the purpose of the consultation is to give anyone with information on PFAS the opportunity to have their say. Of particular interest is information relevant to the risks, socio-economic aspects, and alternative substances. ECHA’s scientific committees for Risk Assessment (RAC) and for Socio-Economic Analysis (SEAC) will use the consultation input to evaluate the proposed restriction and to form an opinion on it.  However, due to the rather high volume of the information that must be collected and analysed with regard to the economic and health impacts of the proposed restriction, we request a 6-month extension to the consultation period, in order to gather detailed data on these points."""

In [16]:
def few_grouped(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:


Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: In favor

Example 3:
Comment 3: {comment_3}
Stance 3: In favor

Example 4:
Comment 4: {comment_4}
Stance 4: Against

Example 5:
Comment 5: {comment_5}
Stance 5: Against

Example 6
Comment 6: {comment_6}
Stance 6: Against

Example 7:
Comment 7: {comment_7}
Stance 7: Neutral

Example 8:
Comment 8: {comment_8}
Stance 8: Neutral

Example 9:
Comment 9: {comment_9}
Stance 9: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:


Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [18]:
def get_response(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" Error: {e}")
        return "error"

In [18]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["cleaned_text"])

    # STANCE CLASSIFICATION
    if not row["Stance"]:
        stance = get_response(few_grouped(comment))
        df.at[idx, "Stance"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/4771 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   0%|          | 1/4771 [00:07<9:37:57,  7.27s/it]

Checkpoint saved at row 0
[1] Stance → Neutral


Classifying comments:   0%|          | 2/4771 [00:09<5:29:51,  4.15s/it]

...

Classifying comments: 100%|█████████▉| 4770/4771 [3:32:34<00:02,  2.17s/it]

[4770] Stance → Against


Classifying comments: 100%|██████████| 4771/4771 [3:32:42<00:00,  2.67s/it]

Checkpoint saved at row 4770


 Classification completed and saved.


# Exemption Classification

## Few-shot classification: grouped classes

In [12]:
input_path = "/content/drive/MyDrive/pfas_stances.xlsx"
output_path = "/content/drive/MyDrive/pfas_stance_exemption.xlsx"
checkpoint_interval = 5

In [13]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "Exemption"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Starting from input


In [14]:
comment_1 = """Avery Dennison Corporation is a global materials science and digital identification solutions company that provides branding and information labeling solutions, including pressure-sensitive materials, radio-frequency identification (RFID) inlays and tags, and a variety of converted products and solutions. The company designs and manufactures a wide range of labeling and functional materials that enhance branded packaging, carry or display information that connects the physical and the digital, and improve customers’ product performance. The company serves an array of industries worldwide, including home and personal care, apparel, e-commerce, logistics, food and grocery, pharmaceuticals and automotive.  PFAS substances and materials are currently being phased out of the Avery Dennison portfolio,  which is consistent with the implementation of the PFAS restriction process.  We have, however, identified two uses where technological alternatives are currently not available, hence the request for a time limited derogation to engineer a substitute and phase out. For both these applications the components containing PFAS are purchased externally.  Use sector: Transport  Sub-use: Use of PFASs in applications affecting the proper functioning related to the safety of vehicles, and affecting the safety of operators, passengers or goods, to the extent not addressed under other parts of this proposed restriction. 9034 Information submitted confidentially 9034    The two missing uses identified are listed below and fall within the use sector "Transport". An assessment of alternatives and substitution timelines are outlined in detail in the confidential section of the submission.    1)Protective Overlay Films for Traffic Signs and Public Transport vehicles - These products are essential for the protection and durability of road signs and public transport vehicle signs which must retain their visibility in low-light conditions and withstand the long-term effects of weather and wear. Stringent regulatory standards apply to this use.  2)Release liners for pressure-sensitive silicone-based adhesives for Airbag labels (Sub-use 1) and Brake shims (Sub-Use 2). Avery Dennison uses a fluorosilicone release liner to protect the pressure-sensitive adhesive before its application, protecting it during shipment, storage, and converting. Sub-uses are highly regulated."""
comment_2 = """Water and oil repellent sprays used for shoes and sandals made of leather or canvas must have not only water repellency but also oil repellency from the standpoint of stain prevention. Oily substances such as cooking sauces and salad dressings soak into leather and fibers, causing stains. If leather products or silk Japanese clothes get dirty, cleaning is difficult and time-consuming. In some cases, these stains cannot be removed even by cleaning, and the product cannot be used. In order to avoid such troubles, it is essential to coat the surface with an oil-repellent fluorine-based substance.  Various resins such as acrylic and urethane resins, as well as silicone-based compounds, can be considered as substitutes for PFAS in this application, but compounds other than fluorine-based compounds do not have oil repellency and cannot prevent stains. In addition, it cannot be used as a substitute for this application because of its poor water repellency. 4121 Leather and textile protection (oil  and water repellent) 4121 There are no PFAS emissions during manufacturing and use. At the end of its useful life, the entire amount is disposed of, but almost all of it is incinerated and decomposed into HF, so no PFAS is released into the environment. 4121 At the end of its useful life, the entire amount is disposed of, but almost all of it is incinerated and decomposed into HF, so no PFAS is released into the environment. 4121  If we limit it to leather and silk products, we estimate that it is 100 to 500 tons each year. 4121    There is no academic data that it is a precursor of these substances. """
comment_3 = """We apply for exemption of following PFAS substances used in plastic materials and/or lubricants of our products POLYTETRAFLUOROETHYLENE (CAS No: 9002-84-0) TETRAFLUOROETHYLENE-HEXAFLUOROPROPYLENE COPOLYMER (CAS No: 25067-11-2) TETRAFLUOROETHYLENE (CAS No: 116-14-3) ETHENE,1,1,2,2-TETRAFLUORO-, OXIDIZED, POLYMD. (CAS No: 69991-61-3) POLY(PERFLUOROPROPYLENE OXIDE-CO-PERFLUOROFORMALDEHYDE) (CAS No: 69991-67-9) 1,1,2,3,3,3-HEXAFLUORO-1-PROPENE, OXIDIZED, POLYERMIZED (CAS No: 161075-14-5) FLUORINERT (CAS No: 86508-42-1) PERFLUOROALKYLETHER (CAS No: 60164-51-4) PERFLUOROOCTANE SULFONATE (PFOS) (CAS No: 1763-23-1) 7726 Electronics and Semiconductors-Electronic component 7726 These PFAS substances are only present in the plastic and/or lubricants of our electronic connector parts. We don't do material recycling and the end of life stage will depend on the end customer. 7726 These PFAS substances are only present in the plastic and/or lubricants of our electronic connector parts. We don't do material recycling and the end of life stage will depend on the end customer 7726 These PFAS substances are only present in the plastic and/or lubricants of our electronic connector parts. We don't do material recycling and the end of life stage will depend on the end customer. 7726 We have a material replacement plan in place and will use material containing PFAS substances 7726 We have a material replacement plan in place and will use material containing PFAS substances 7726 We have a material replacement plan in place and will use material containing PFAS substances 7726 We have a material replacement plan in place and will use material containing PFAS substances."""


comment_4 = """The proposal is good as it is and must not be watered down. I am willing to sacrifice certain characteristics of products, such as heavier phones or less waterproof rain jackets, if it means we can stop using PFAS. There is more and more research showing how bad this is for all of us. We will be ashamed and regretful in the future for not stopping this when we could. Please listen to the public and not just the companies!"""
comment_5 = """REINTJES Benelux BV appreciates the opportunity to provide the following comments in response to the proposal of this PFAS restriction. REINTJES supports the EU's efforts to protect human health and environment. However, this proposal does not seem to adequately address the impact on a wide range of industries. For this reason, we comment on this proposal as follows. A blanket restriction of the entire PFAS substance group without a differentiated, substance- and application-specific risk assessment and solely due to the persistence of some PFAS is not appropriate. In order to ultimately achieve a sustainable overall balance of resource conservation and environmental impact,  a restriction is only justified in cases where the risks to humans and the environment cannot be controlled. It should be taken into account that the PFAS definition includes substances with different properties and that neither all PFAS are equally persistent. This puts almost risk-free chemicals on an equal footing with substances of very high concern with properties that require regulation. As part of a differentiated approach, it is urgent to ensure that only those substances whose use poses an unacceptable risk to the environment or human health are banned. Otherwise, there is a risk that chemicals that play a crucial role in innovative technologies will be driven out of the market. For example, manufacturers of hydraulic components such as pumps, motors, valves and cylinders as well as manufacturers of valves and compressors are affected. PFAS, mostly fluorinated polymers, are often used in seals, hoses, pipes, valves and coatings that we absolutely need for our products. While in some cases "only" the performance of some products would be massively affected, some products could no longer be manufactured, which would mean a very high impact on not only our company, our customers and our market but the entire shipping industry. 8735 1. Sectors and (sub-)uses: - Electronics and semiconductor (Annex E.2.11.) - Construction products (Annex E.2.13.) - Lub"""
comment_6 = """General comments: Please refer to the attached file in Section V. 7108 1. Sectors and (sub-)uses: Sectors :      Transport (Sub-)Uses : Use of PFASs in applications affecting the proper functioning related to the safety of vehicles, and                       affecting the safety  of  operators, passengers or  goods, to  the extent not addressed under other                      parts of this proposed restriction. 7108 2. Emissions in the end-of-life phase: Please refer to the attached file in Section V. 7108 7. Potential derogations marked for reconsideration: Please refer to the attached files in Section V. 7108 8. Other identified uses: Please refer to the attached files in Section V."""

In [15]:
def few_grouped(comment):
    return f"""
The task is to determine whether a public consultation comment explicitly or implicitly requests an exemption or derogation from the proposed PFAS restriction policy.

You are a language model trained to identify whether a stakeholder is requesting an exemption in a public consultation comment regarding the PFAS restriction proposal.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Label 1: 1

Example 2:
Comment 2: {comment_2}
Label 2: 1

Example 3:
Comment 3: {comment_3}
Label 3: 1

Example 4:
Comment 4: {comment_4}
Label 4: 0

Example 5:
Comment 5: {comment_5}
Label 5: 0

Example 6:
Comment 6: {comment_6}
Label 6: 0

Taking the examples into consideration, determine whether the following comment requests any exemption or derogation from the PFAS restriction policy:

Comment:
\"{comment}\"

Respond with only one of the following digits:
- 1 → if an exemption is requested
- 0 → if no exemption is requested

Answer with only 1 or 0.
"""


In [19]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["cleaned_text"])

    # STANCE CLASSIFICATION
    if not row["Exemption"]:
        stance = get_response(few_grouped(comment))
        df.at[idx, "Exemption"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/4771 [00:00<?, ?it/s]

[0] Stance → 1


Classifying comments:   0%|          | 1/4771 [00:08<11:30:10,  8.68s/it]

Checkpoint saved at row 0
[1] Stance → 0


Classifying comments:   0%|          | 2/4771 [00:11<6:33:31,  4.95s/it] 

...

Classifying comments: 100%|█████████▉| 4770/4771 [3:47:39<00:01,  1.99s/it]

[4770] Stance → 1


Classifying comments: 100%|██████████| 4771/4771 [3:47:46<00:00,  2.86s/it]

Checkpoint saved at row 4770


 Classification completed and saved.
